# TASK 5: COLLABORATIVE FILTERING - SVD

In [1]:
import pandas as pd
import numpy as np
import pickle
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split, GridSearchCV
from surprise import accuracy
import warnings
warnings.filterwarnings('ignore')
print("TASK 5: COLLABORATIVE FILTERING - SVD")

TASK 5: COLLABORATIVE FILTERING - SVD


## CHUẨN BỊ RATING MATRIX

In [2]:
print("\nĐang chuẩn bị Rating Matrix...")

# Load cleaned ratings
ratings = pd.read_csv('../data/cleaned/ratings_cleaned.csv')
movies = pd.read_csv('../data/cleaned/movies_cleaned.csv')

print(f"Đã load ratings: {len(ratings):,} đánh giá")
print(f"Đã load movies: {len(movies):,} bộ phim")

print(f"\nThống kê Ratings:")
print(f"   - Số users: {ratings['userId'].nunique():,}")
print(f"   - Số movies: {ratings['movieId'].nunique():,}")
print(f"   - Số ratings: {len(ratings):,}")
print(f"   - Rating range: {ratings['rating'].min():.1f} - {ratings['rating'].max():.1f}")
print(f"   - Rating trung bình: {ratings['rating'].mean():.2f}")

# Prepare data for Surprise library
print("\nĐang chuẩn bị data cho Surprise library...")
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Split train/test (80/20)
print("\nĐang split train/test (80/20)...")
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"Trainset: {trainset.n_ratings:,} ratings")
print(f"Testset: {len(testset):,} ratings")


Đang chuẩn bị Rating Matrix...
Đã load ratings: 1,000,209 đánh giá
Đã load movies: 3,416 bộ phim

Thống kê Ratings:
   - Số users: 6,040
   - Số movies: 3,706
   - Số ratings: 1,000,209
   - Rating range: 1.0 - 5.0
   - Rating trung bình: 3.58

Đang chuẩn bị data cho Surprise library...

Đang split train/test (80/20)...
Trainset: 800,167 ratings
Testset: 200,042 ratings


## TRAIN SVD MODEL (BASELINE)

In [3]:
print("TRAIN SVD MODEL (BASELINE)")

print("\nKhởi tạo SVD model với hyperparameters mặc định...")
print("   - n_factors: 100")
print("   - n_epochs: 20")
print("   - lr_all: 0.005")
print("   - reg_all: 0.02")

# Train baseline model
svd_baseline = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42,
    verbose=True
)

print("\nĐang train model...")
svd_baseline.fit(trainset)
print("Training hoàn tất!")

# Evaluate baseline
print("\nĐánh giá Baseline Model trên Test Set:")
predictions_baseline = svd_baseline.test(testset)
rmse_baseline = accuracy.rmse(predictions_baseline, verbose=False)
mae_baseline = accuracy.mae(predictions_baseline, verbose=False)

print(f"   - RMSE: {rmse_baseline:.4f}")
print(f"   - MAE: {mae_baseline:.4f}")


TRAIN SVD MODEL (BASELINE)

Khởi tạo SVD model với hyperparameters mặc định...
   - n_factors: 100
   - n_epochs: 20
   - lr_all: 0.005
   - reg_all: 0.02

Đang train model...
Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Training hoàn tất!

Đánh giá Baseline Model trên Test Set:
   - RMSE: 0.8740
   - MAE: 0.6858


## HYPERPARAMETER TUNING (OPTIMIZED)

In [4]:
print("HYPERPARAMETER TUNING (OPTIMIZED GRID SEARCH)")

print("\nĐang setup GridSearchCV...")

# Optimized parameter grid
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [10, 20, 30],
    'lr_all': [0.005],     
    'reg_all': [0.02]       
   
}

print(f"\nParameter Grid:")
print(f"   - n_factors: {param_grid['n_factors']}")
print(f"   - n_epochs: {param_grid['n_epochs']}")
print(f"   - lr_all: {param_grid['lr_all']}")  
print(f"   - reg_all: {param_grid['reg_all']}") 
print(f"   → Tổng số combinations: {len(param_grid['n_factors']) * len(param_grid['n_epochs']) * len(param_grid['lr_all']) * len(param_grid['reg_all'])}")

# GridSearchCV với 3-fold CV
gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1,  # Sử dụng tất cả CPU cores
    joblib_verbose=2
)

print("\nĐang chạy Grid Search (có thể mất 8-12 phút)...")
print("   (Sử dụng tất cả CPU cores để tăng tốc...)")

gs.fit(data)

print("\nGrid Search hoàn tất!")

# Best parameters
print("\nBest Parameters (RMSE):")
print(f"   - n_factors: {gs.best_params['rmse']['n_factors']}")
print(f"   - n_epochs: {gs.best_params['rmse']['n_epochs']}")
print(f"   - lr_all: {gs.best_params['rmse']['lr_all']}")      
print(f"   - reg_all: {gs.best_params['rmse']['reg_all']}")    
print(f"   - Best RMSE: {gs.best_score['rmse']:.4f}")

print("\nBest Parameters (MAE):")
print(f"   - n_factors: {gs.best_params['mae']['n_factors']}")
print(f"   - n_epochs: {gs.best_params['mae']['n_epochs']}")
print(f"   - lr_all: {gs.best_params['mae']['lr_all']}")      
print(f"   - reg_all: {gs.best_params['mae']['reg_all']}") 
print(f"   - Best MAE: {gs.best_score['mae']:.4f}")

# Train final model với best parameters
print("\nĐang train Final Model với best parameters...")
best_params = gs.best_params['rmse']

svd_final = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],  # Fixed
    reg_all=best_params['reg_all'],  # Fixed
    random_state=42,
    verbose=True
)

svd_final.fit(trainset)
print("Training Final Model hoàn tất!")


HYPERPARAMETER TUNING (OPTIMIZED GRID SEARCH)

Đang setup GridSearchCV...

Parameter Grid:
   - n_factors: [50, 100, 150]
   - n_epochs: [10, 20, 30]
   - lr_all: [0.005]
   - reg_all: [0.02]
   → Tổng số combinations: 9

Đang chạy Grid Search (có thể mất 8-12 phút)...
   (Sử dụng tất cả CPU cores để tăng tốc...)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 out of  27 | elapsed:   27.7s remaining:   47.2s
[Parallel(n_jobs=-1)]: Done  24 out of  27 | elapsed:   58.4s remaining:    7.2s
[Parallel(n_jobs=-1)]: Done  27 out of  27 | elapsed:  1.1min finished



Grid Search hoàn tất!

Best Parameters (RMSE):
   - n_factors: 50
   - n_epochs: 20
   - lr_all: 0.005
   - reg_all: 0.02
   - Best RMSE: 0.8829

Best Parameters (MAE):
   - n_factors: 50
   - n_epochs: 20
   - lr_all: 0.005
   - reg_all: 0.02
   - Best MAE: 0.6944

Đang train Final Model với best parameters...
Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Training Final Model hoàn tất!


## ĐÁNH GIÁ FINAL MODEL

In [5]:
print("ĐÁNH GIÁ FINAL MODEL")

print("\nĐánh giá trên Test Set:")
predictions_final = svd_final.test(testset)
rmse_final = accuracy.rmse(predictions_final, verbose=False)
mae_final = accuracy.mae(predictions_final, verbose=False)

print(f"   - RMSE: {rmse_final:.4f}")
print(f"   - MAE: {mae_final:.4f}")

print("\nSo sánh Baseline vs Final Model:")
print(f"{'Metric':<10} {'Baseline':<12} {'Final Model':<12} {'Improvement':<12}")
print("-" * 50)
print(f"{'RMSE':<10} {rmse_baseline:<12.4f} {rmse_final:<12.4f} {(rmse_baseline - rmse_final)/rmse_baseline * 100:>10.2f}%")
print(f"{'MAE':<10} {mae_baseline:<12.4f} {mae_final:<12.4f} {(mae_baseline - mae_final)/mae_baseline * 100:>10.2f}%")

ĐÁNH GIÁ FINAL MODEL

Đánh giá trên Test Set:
   - RMSE: 0.8713
   - MAE: 0.6841

So sánh Baseline vs Final Model:
Metric     Baseline     Final Model  Improvement 
--------------------------------------------------
RMSE       0.8740       0.8713             0.31%
MAE        0.6858       0.6841             0.26%


## BUILD RECOMMENDATION FUNCTION

In [6]:
print("XÂY DỰNG HÀM GỢI Ý")

def svd_recommend(user_id, n=10, verbose=True):
    """
    Gợi ý phim cho user dựa trên SVD Collaborative Filtering
    
    Parameters:
    -----------
    user_id : int
        ID của user cần gợi ý
    n : int
        Số lượng phim gợi ý (default=10)
    verbose : bool
        Hiển thị thông tin chi tiết
    
    Returns:
    --------
    DataFrame: Danh sách phim gợi ý
    """
    
    # Kiểm tra user có tồn tại không
    if user_id not in ratings['userId'].values:
        print(f"User ID {user_id} không tồn tại trong dataset")
        return None
    
    # Lấy danh sách phim user đã rate
    user_ratings = ratings[ratings['userId'] == user_id]
    rated_movies = user_ratings['movieId'].values
    
    if verbose:
        print(f"\nUser ID: {user_id}")
        print(f"   - Đã đánh giá: {len(rated_movies)} phim")
        print(f"   - Rating trung bình: {user_ratings['rating'].mean():.2f}")
    
    # Lấy danh sách tất cả movies chưa rate
    all_movies = movies['movieId'].values
    unrated_movies = [m for m in all_movies if m not in rated_movies]
    
    if verbose:
        print(f"   - Số phim chưa rate: {len(unrated_movies)}")
    
    # Predict rating cho tất cả unrated movies
    if verbose:
        print(f"\nĐang predict ratings cho {len(unrated_movies)} phim...")
    
    predictions = []
    for movie_id in unrated_movies:
        pred = svd_final.predict(user_id, movie_id)
        predictions.append({
            'movieId': movie_id,
            'predicted_rating': pred.est
        })
    
    # Convert to DataFrame và sort
    pred_df = pd.DataFrame(predictions)
    pred_df = pred_df.sort_values('predicted_rating', ascending=False)
    
    # Lấy top N
    top_n = pred_df.head(n)
    
    # Merge với thông tin phim
    recommendations = top_n.merge(
        movies[['movieId', 'title_clean', 'genres', 'rating_avg', 'rating_count']], 
        on='movieId'
    )
    recommendations['rank'] = range(1, len(recommendations) + 1)
    
    # Sắp xếp columns
    result_cols = ['rank', 'movieId', 'title_clean', 'genres', 
                   'predicted_rating', 'rating_avg', 'rating_count']
    recommendations = recommendations[result_cols]
    
    if verbose:
        print(f"\nTop {n} phim gợi ý cho User {user_id}:")
        print("-" * 110)
        for _, row in recommendations.iterrows():
            print(f"#{row['rank']:<2} | {row['title_clean']:<35} | {row['genres']:<25} | "
                  f"Pred: {row['predicted_rating']:.2f} | Avg: {row['rating_avg']:.2f}")
    
    return recommendations


def svd_recommend_for_new_user(favorite_movies, n=10, verbose=True):
    """
    Gợi ý phim cho user mới dựa trên phim yêu thích (Cold Start)
    
    Parameters:
    -----------
    favorite_movies : list of tuples
        List các (movie_id, rating) của user
        Example: [(1, 5), (50, 4), (260, 5)]
    n : int
        Số lượng phim gợi ý
    verbose : bool
        Hiển thị thông tin chi tiết
    
    Returns:
    --------
    DataFrame: Danh sách phim gợi ý
    """
    
    # Tạo temporary user ID
    temp_user_id = ratings['userId'].max() + 1
    
    if verbose:
        print(f"\nUser mới (temp ID: {temp_user_id})")
        print(f"   - Số phim yêu thích: {len(favorite_movies)}")
    
    # Tạo trainset mới với ratings của user mới
    temp_ratings = ratings.copy()
    
    for movie_id, rating in favorite_movies:
        temp_ratings = pd.concat([
            temp_ratings,
            pd.DataFrame([{'userId': temp_user_id, 'movieId': movie_id, 'rating': rating}])
        ], ignore_index=True)
    
    # Train model mới (hoặc dùng model cũ với refit)
    reader = Reader(rating_scale=(1, 5))
    temp_data = Dataset.load_from_df(temp_ratings[['userId', 'movieId', 'rating']], reader)
    temp_trainset = temp_data.build_full_trainset()
    
    if verbose:
        print(f"\nĐang retrain model với ratings của user mới...")
    
    # Clone model và train
    temp_model = SVD(
        n_factors=best_params['n_factors'],
        n_epochs=best_params['n_epochs'],
        lr_all=best_params['lr_all'],
        reg_all=best_params['reg_all'],
        random_state=42,
        verbose=False
    )
    temp_model.fit(temp_trainset)
    
    # Lấy danh sách movies chưa rate
    rated_movies = [m for m, r in favorite_movies]
    unrated_movies = [m for m in movies['movieId'].values if m not in rated_movies]
    
    # Predict
    predictions = []
    for movie_id in unrated_movies:
        pred = temp_model.predict(temp_user_id, movie_id)
        predictions.append({
            'movieId': movie_id,
            'predicted_rating': pred.est
        })
    
    # Convert to DataFrame và lấy top N
    pred_df = pd.DataFrame(predictions)
    pred_df = pred_df.sort_values('predicted_rating', ascending=False).head(n)
    
    # Merge với thông tin phim
    recommendations = pred_df.merge(
        movies[['movieId', 'title_clean', 'genres', 'rating_avg', 'rating_count']], 
        on='movieId'
    )
    recommendations['rank'] = range(1, len(recommendations) + 1)
    
    result_cols = ['rank', 'movieId', 'title_clean', 'genres', 
                   'predicted_rating', 'rating_avg', 'rating_count']
    recommendations = recommendations[result_cols]
    
    if verbose:
        print(f"\nTop {n} phim gợi ý:")
        print("-" * 110)
        for _, row in recommendations.iterrows():
            print(f"#{row['rank']:<2} | {row['title_clean']:<35} | {row['genres']:<25} | "
                  f"Pred: {row['predicted_rating']:.2f} | Avg: {row['rating_avg']:.2f}")
    
    return recommendations


print("Đã tạo 2 functions:")
print("   1. svd_recommend(user_id, n=10)")
print("   2. svd_recommend_for_new_user(favorite_movies, n=10)")


XÂY DỰNG HÀM GỢI Ý
Đã tạo 2 functions:
   1. svd_recommend(user_id, n=10)
   2. svd_recommend_for_new_user(favorite_movies, n=10)


## TEST RECOMMENDATION FUNCTIONS

In [7]:
print("KIỂM TRA HÀM GỢI Ý")

# TEST 1: Gợi ý cho user có sẵn
print("TEST 1: GỢI Ý CHO USER CÓ SẴN")

# Lấy một user ngẫu nhiên có nhiều ratings
active_users = ratings['userId'].value_counts()
test_user_id = active_users[active_users > 50].sample(1).index[0]

print(f"\nTest với User ID: {test_user_id}")
recs_1 = svd_recommend(test_user_id, n=10, verbose=True)

# Hiển thị một vài phim user đã rate để so sánh
print(f"\nMột số phim user đã rate (để tham khảo):")
user_rated = ratings[ratings['userId'] == test_user_id].merge(
    movies[['movieId', 'title_clean', 'genres']], on='movieId'
).sort_values('rating', ascending=False).head(5)

for _, row in user_rated.iterrows():
    print(f"{row['rating']:.1f} | {row['title_clean']:<40} | {row['genres']}")

# TEST 2: Gợi ý cho user mới (Cold Start)
print("TEST 2: GỢI Ý CHO USER MỚI (COLD START)")

# Giả sử user mới thích 3 phim
favorite_movies = [
    (1, 5),    # Toy Story
    (260, 4),  # Star Wars
    (1196, 5)  # Star Wars: Empire Strikes Back
]

print(f"\nUser mới yêu thích:")
for movie_id, rating in favorite_movies:
    movie_info = movies[movies['movieId'] == movie_id].iloc[0]
    print(f"{rating} | {movie_info['title_clean']} ({movie_info['genres']})")

recs_2 = svd_recommend_for_new_user(favorite_movies, n=10, verbose=True)

KIỂM TRA HÀM GỢI Ý
TEST 1: GỢI Ý CHO USER CÓ SẴN

Test với User ID: 1115

User ID: 1115
   - Đã đánh giá: 109 phim
   - Rating trung bình: 3.17
   - Số phim chưa rate: 3307

Đang predict ratings cho 3307 phim...

Top 10 phim gợi ý cho User 1115:
--------------------------------------------------------------------------------------------------------------
#1  | Usual Suspects, The                 | Crime|Thriller            | Pred: 4.57 | Avg: 4.52
#2  | Magnolia                            | Drama                     | Pred: 4.47 | Avg: 3.89
#3  | North by Northwest                  | Drama|Thriller            | Pred: 4.35 | Avg: 4.38
#4  | Fight Club                          | Drama                     | Pred: 4.35 | Avg: 4.08
#5  | Seven Samurai (The Magnificent Seven) (Shichinin no samurai) | Action|Drama              | Pred: 4.31 | Avg: 4.56
#6  | Princess Bride, The                 | Action|Adventure|Comedy|Romance | Pred: 4.30 | Avg: 4.30
#7  | Matrix, The                         

## LƯU MODEL

In [8]:
print("LƯU SVD MODEL")

# Save SVD model
with open('../models/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_final, f)
print("Đã lưu: models/svd_model.pkl")

# Save training info
training_info = {
    'best_params': gs.best_params['rmse'],
    'best_rmse': gs.best_score['rmse'],
    'best_mae': gs.best_score['mae'],
    'baseline_rmse': rmse_baseline,
    'baseline_mae': mae_baseline,
    'final_rmse': rmse_final,
    'final_mae': mae_final,
    'n_users': ratings['userId'].nunique(),
    'n_movies': ratings['movieId'].nunique(),
    'n_ratings': len(ratings),
    'train_size': trainset.n_ratings,
    'test_size': len(testset)
}

with open('../models/svd_training_info.pkl', 'wb') as f:
    pickle.dump(training_info, f)
print("Đã lưu: models/svd_training_info.pkl")

# Save GridSearch results
results_df = pd.DataFrame(gs.cv_results)
results_df.to_csv('../models/svd_gridsearch_results.csv', index=False)
print("Đã lưu: models/svd_gridsearch_results.csv")

LƯU SVD MODEL
Đã lưu: models/svd_model.pkl
Đã lưu: models/svd_training_info.pkl
Đã lưu: models/svd_gridsearch_results.csv


## TỔNG KẾT

In [9]:
print("TASK 5 HOÀN THÀNH!")

print("\nChecklist:")
print("Chuẩn bị Rating Matrix")
print("Train SVD Baseline Model")
print("Hyperparameter Tuning (Optimized Grid Search)")
print("Train Final Model với best params")
print("Build recommendation functions")
print("Test với nhiều trường hợp")
print("Lưu model và results")

print("\nFinal Model Performance:")
print(f"RMSE: {rmse_final:.4f}")
print(f"MAE: {mae_final:.4f}")
print(f"Improvement vs Baseline: {(rmse_baseline - rmse_final)/rmse_baseline * 100:.2f}%")

print("\nBest Hyperparameters:")
print(f"n_factors: {best_params['n_factors']}")
print(f"n_epochs: {best_params['n_epochs']}")
print(f"lr_all: 0.005 (fixed)")
print(f"reg_all: 0.02 (fixed)")

print("\nOutput files:")
print("  - models/svd_model.pkl")
print("  - models/svd_training_info.pkl")
print("  - models/svd_gridsearch_results.csv")

print("\nFunctions available:")
print("  - svd_recommend(user_id, n=10)")
print("  - svd_recommend_for_new_user(favorite_movies, n=10)")

TASK 5 HOÀN THÀNH!

Checklist:
Chuẩn bị Rating Matrix
Train SVD Baseline Model
Hyperparameter Tuning (Optimized Grid Search)
Train Final Model với best params
Build recommendation functions
Test với nhiều trường hợp
Lưu model và results

Final Model Performance:
RMSE: 0.8713
MAE: 0.6841
Improvement vs Baseline: 0.31%

Best Hyperparameters:
n_factors: 50
n_epochs: 20
lr_all: 0.005 (fixed)
reg_all: 0.02 (fixed)

Output files:
  - models/svd_model.pkl
  - models/svd_training_info.pkl
  - models/svd_gridsearch_results.csv

Functions available:
  - svd_recommend(user_id, n=10)
  - svd_recommend_for_new_user(favorite_movies, n=10)
